# Same pixels, new picture

Generate an image from a text prompt whose color histogram is **identical** to a source image's. Since the output is the same size, that means it's literally a rearrangement of the source's pixels.

**How it works**

1. Sample from Stable Diffusion 3.5 Medium (a rectified-flow model) with a custom loop.
2. In the middle of sampling, take the model's prediction of the final image, pull its colors onto the source's color distribution (*sliced optimal transport*), and put it back on the same noise trajectory. The model then has the remaining steps to make the color-corrected image look natural.
3. At the end, an exact snap assigns every output pixel a distinct source pixel, so the histogram matches bit for bit.

**Before running**

- *Runtime → Change runtime type → GPU.* A T4 works at 512×512.
- Accept the license at [huggingface.co/stabilityai/stable-diffusion-3.5-medium](https://huggingface.co/stabilityai/stable-diffusion-3.5-medium) and have a Hugging Face access token ready.

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Load the model

In [ ]:
import torch, numpy as np
from diffusers import StableDiffusion3Pipeline

device = "cuda"
# bf16 needs an Ampere-or-newer GPU (L4, A100). A T4 falls back to fp16.
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Skipping the T5 text encoder saves ~9GB of GPU memory. Prompts still work
# through SD3's two CLIP encoders, just with a bit less prompt adherence.
pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    text_encoder_3=None, tokenizer_3=None, torch_dtype=dtype,
).to(device)
vae = pipe.vae

## Settings

In [ ]:
SIZE = 512                                              # output is SIZE x SIZE
PROMPT = "a snowy mountain range at dawn, photograph"   # what to draw
SOURCE_PROMPT = "a tropical beach at sunset, photograph" # source image, if you don't upload one
STEPS = 28
CFG = 4.5
SEED = 0

## Source image

The pixels you're allowed to use. Upload your own, or generate one.

In [ ]:
from PIL import Image, ImageOps

UPLOAD = False   # True: upload your own image. False: generate one from SOURCE_PROMPT.

if UPLOAD:
    from google.colab import files
    name = next(iter(files.upload()))
    source = Image.open(name).convert("RGB")
else:
    source = pipe(SOURCE_PROMPT, height=SIZE, width=SIZE, num_inference_steps=STEPS,
                  guidance_scale=CFG, generator=torch.Generator(device).manual_seed(123)).images[0]

source = ImageOps.fit(source, (SIZE, SIZE))   # center-crop so pixel counts match
src8 = np.asarray(source)                     # (H, W, 3) uint8
src = torch.tensor(src8, device=device, dtype=torch.float32).reshape(-1, 3) / 255
source

## Helpers

- `decode` / `encode` convert between SD3's latent space and pixels. The histogram lives in pixel space, so every projection is a round trip.
- `sliced_ot` is the projection: rotate RGB space randomly, match each axis exactly by sorting, rotate back, repeat. Each pixel keeps its *rank* along a direction but takes the source's value at that rank.

In [ ]:
def decode(lat):
    # latents -> image in [0, 1], shape (1, 3, H, W)
    lat = lat / vae.config.scaling_factor + vae.config.shift_factor
    return ((vae.decode(lat.to(vae.dtype)).sample.float() + 1) / 2).clamp(0, 1)

def encode(img):
    # image in [0, 1], shape (1, 3, H, W) -> latents
    lat = vae.encode((img * 2 - 1).to(vae.dtype)).latent_dist.mean
    return (lat - vae.config.shift_factor) * vae.config.scaling_factor

def to_pixels(img):    # (1, 3, H, W) -> (H*W, 3)
    return img[0].permute(1, 2, 0).reshape(-1, 3)

def from_pixels(px):   # (H*W, 3) -> (1, 3, H, W)
    return px.reshape(SIZE, SIZE, 3).permute(2, 0, 1)[None]

def to_uint8(img):     # (1, 3, H, W) in [0, 1] -> (H, W, 3) uint8
    return (img[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)

def sliced_ot(x, src, iters=30):
    # Move x's color distribution onto src's. x, src: (n, 3) float tensors.
    x = x.clone()
    for _ in range(iters):
        Q, _ = torch.linalg.qr(torch.randn(3, 3, device=x.device))   # random rotation of RGB space
        xp, sp = x @ Q, src @ Q
        for k in range(3):
            order = xp[:, k].argsort()
            xp[order, k] = sp[:, k].sort().values   # i-th smallest takes src's i-th smallest
        x = xp @ Q.T
    return x

## Exact snap

Makes the output literally a permutation of the source's pixels:

1. Pixels whose color exists in the source (and hasn't been used up) take that exact pixel, with zero change.
2. The leftovers are paired by sorting both sets along a 3D Hilbert curve through color space, which keeps nearby colors nearby.

In [ ]:
def color_keys(c):
    c = c.astype(np.int64)
    return (c[:, 0] << 16) | (c[:, 1] << 8) | c[:, 2]

def hilbert(c, bits=8):
    # Position of each RGB color along a 3D Hilbert curve (Skilling's algorithm, vectorized).
    X = [c[:, i].astype(np.int64).copy() for i in range(3)]
    M = 1 << (bits - 1); Q = M
    while Q > 1:
        P = Q - 1
        for i in range(3):
            m = (X[i] & Q) != 0
            X[0] = np.where(m, X[0] ^ P, X[0])
            t = np.where(m, 0, (X[0] ^ X[i]) & P)
            X[0] ^= t; X[i] ^= t
        Q >>= 1
    for i in range(1, 3): X[i] ^= X[i - 1]
    t = np.zeros_like(X[0]); Q = M
    while Q > 1:
        t = np.where((X[2] & Q) != 0, t ^ (Q - 1), t); Q >>= 1
    for i in range(3): X[i] ^= t
    key = np.zeros(len(c), np.int64)
    for b in range(bits - 1, -1, -1):
        for i in range(3):
            key = (key << 1) | ((X[i] >> b) & 1)
    return key

def occurrence_ids(k):
    # Unique id per pixel: (color, how many times that color appeared before it).
    n = len(k); order = np.argsort(k, kind="stable"); ks = k[order]
    starts = np.r_[0, np.flatnonzero(np.diff(ks)) + 1]
    counts = np.diff(np.r_[starts, n])
    rank = np.empty(n, np.int64); rank[order] = np.arange(n) - np.repeat(starts, counts)
    return k * n + rank

def exact_snap(img, src):
    # Rearrange src's pixels to look as much like img as possible. Both (H, W, 3) uint8.
    x, s = img.reshape(-1, 3), src.reshape(-1, 3); n = len(s)
    ix, is_ = occurrence_ids(color_keys(x)), occurrence_ids(color_keys(s))
    s_order = np.argsort(is_); s_sorted = is_[s_order]
    pos = np.minimum(np.searchsorted(s_sorted, ix), n - 1)
    hit = s_sorted[pos] == ix
    perm = np.full(n, -1); perm[hit] = s_order[pos[hit]]
    used = np.zeros(n, bool); used[perm[hit]] = True
    lx, ls = np.flatnonzero(~hit), np.flatnonzero(~used)
    perm[lx[np.argsort(hilbert(x[lx]))]] = ls[np.argsort(hilbert(s[ls]))]
    return s[perm].reshape(src.shape), hit.mean()

def same_histogram(a, b):
    return np.array_equal(np.sort(color_keys(a.reshape(-1, 3))), np.sort(color_keys(b.reshape(-1, 3))))

## The sampling loop

`diffusers` runs time from noise (σ = 1) to image (σ = 0), with `x_t = (1 − σ)·image + σ·noise`. The model predicts the velocity `v = noise − image`, so at every step we can recover both ends of the path:

```
predicted image  x̂₀    = x_t − σ·v
starting noise   noise = x_t + (1 − σ)·v
```

The projection replaces `x̂₀` with its color-corrected version and rebuilds `x_t` from the same noise. It only runs inside `window`: too early and `x̂₀` is a blurry average, too late and the model has no steps left to make the corrected image look natural.

In [ ]:
@torch.no_grad()
def generate(prompt, project=True, project_every=3, window=(0.15, 0.9), seed=SEED):
    pe, npe, ppe, nppe = pipe.encode_prompt(prompt, prompt, prompt, device=device)
    cond, pooled = torch.cat([npe, pe]), torch.cat([nppe, ppe])
    pipe.scheduler.set_timesteps(STEPS, device=device)
    sig = pipe.scheduler.sigmas
    c, f = pipe.transformer.config.in_channels, pipe.vae_scale_factor
    lat = torch.randn(1, c, SIZE // f, SIZE // f, device=device, dtype=dtype,
                      generator=torch.Generator(device).manual_seed(seed))

    for i, t in enumerate(pipe.scheduler.timesteps):
        out = pipe.transformer(hidden_states=torch.cat([lat] * 2), timestep=t.expand(2),
                               encoder_hidden_states=cond, pooled_projections=pooled,
                               return_dict=False)[0]
        vu, vc = out.chunk(2)
        v = vu + CFG * (vc - vu)                 # classifier-free guidance
        s = sig[i]

        if project and i % project_every == 0 and window[0] < s < window[1]:
            x0 = lat - s * v                     # where we're headed
            noise = lat + (1 - s) * v            # where we came from
            px = sliced_ot(to_pixels(decode(x0)), src).clamp(0, 1)
            x0 = encode(from_pixels(px)).to(dtype)
            lat = ((1 - s) * x0 + s * noise).to(dtype)   # same noise, corrected target
            v = noise - x0

        lat = (lat + (sig[i + 1] - s) * v).to(dtype)     # Euler step
    return decode(lat)

## Run it

Same prompt and seed, with and without projection, then both snapped to the source's exact pixels.

In [ ]:
baseline = to_uint8(generate(PROMPT, project=False))
projected = to_uint8(generate(PROMPT, project=True))
baseline_snap, b_exact = exact_snap(baseline, src8)
projected_snap, p_exact = exact_snap(projected, src8)

for name, before, after, exact in [("no projection", baseline, baseline_snap, b_exact),
                                   ("projection", projected, projected_snap, p_exact)]:
    change = np.abs(after.astype(int) - before.astype(int)).mean()
    print(f"{name:>14}: snap changed pixels by {change:5.1f}/255 on average, "
          f"{exact:.0%} kept exactly, identical histogram: {same_histogram(after, src8)}")

In [ ]:
import matplotlib.pyplot as plt

panels = [(src8, "source"), (baseline, "no projection"), (baseline_snap, "no projection, snapped"),
          (projected, "projection"), (projected_snap, "projection, snapped")]
fig, axes = plt.subplots(1, 5, figsize=(20, 4.4))
for ax, (img, title) in zip(axes, panels):
    ax.imshow(img); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 3))
for ax, ch, color in zip(axes, range(3), ["red", "green", "blue"]):
    for img, label, style in [(src8, "source", "-"), (projected, "projection", ":"),
                              (projected_snap, "snapped", "--")]:
        ax.plot(np.bincount(img[..., ch].ravel(), minlength=256), style, color=color, label=label, lw=1.2)
    ax.set_title(f"{color} channel"); ax.set_xlim(0, 255); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
for img, name in [(src8, "source"), (projected_snap, "snapped"), (projected, "projected"), (baseline_snap, "baseline_snapped")]:
    Image.fromarray(img).save(f"seed{SEED}_{name}.png")

## Things to try

- **Impossible prompts:** ask for colors the source doesn't have ("a lush green forest" from a sunset). What does the model do with the wrong palette?
- **`window` and `project_every`:** project earlier, later, or every step. Where's the sweet spot between "looks natural" and "snap barely changes anything"?
- **Your own photos:** set `UPLOAD = True`.
- **Swap roles:** generate A from B's pixels, then B from A's.

**If outputs are black or full of noise** on a T4, fp16 has probably overflowed somewhere. Switch the runtime to an L4 or A100 so the notebook uses bf16.